# CRIM Intervals: Melodic and Harmonic Corpus Search
What You Can Do with this Notebook:
Search A Corpus for Melodic and Harmonic nGrams
Note
intervals
https://github.com/HCDigitalScholarship/intervals/

main https://github.com/HCDigitalScholarship/intervals/blob/b125380ae28d9970b2707944cd7f7dfe8720dc4a/crim_intervals/main_objs.py#L3175

verovio
toolkit https://book.verovio.org/toolkit-reference/toolkit-methods.html?q=toolkit

brend measure range issue https://github.com/rism-digital/verovio/issues/1304

A. Import Intervals and Other Code

In [1]:
import crim_intervals
from crim_intervals import * 
from crim_intervals import main_objs
import crim_intervals.visualizations as viz
import pandas as pd
import re
import altair as alt
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact
from ipywidgets import interact, widgets, fixed
from pandas import json_normalize
from pyvis.network import Network
from IPython.display import display
import requests
import os
import glob as glob


MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    print("created folder : ", MYDIR)
else:
    print(MYDIR, "folder already exists.")
    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)
else:
    print(MUSDIR, "folder already exists.")

saved_csv folder already exists.
Music_Files folder already exists.


In [2]:
project_name = input("Project name: ").replace(' ', '_') # to fill
output_folder = os.path.join(project_name, "csv")

if not os.path.exists(project_name):
    os.makedirs(project_name)

Project name:  my


In [4]:
combineUnisons = True # True or False
kind = 'q' # q or d
n = 3 # number

In [5]:
'''
corpus = CorpusBase(['https://crimproject.org/mei/CRIM_Model_0019.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_1.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_2.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_3.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_4.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_5.mei'])
'''
corpus = CorpusBase(['https://crimproject.org/mei/CRIM_Mass_0019_2.mei'])

# GT let's move this to another cell, for clarity
# project_name = "praeparavit_gloria" # to fill
# output_folder = os.path.join(project_name, "csv")

#if not os.path.exists(project_name):
#    os.makedirs(project_name)

# GT move also these to another cell
# combineUnisons = True # True or False
# kind = 'd' # q or d
# n = 4 # number
# GT make this variable interactive, see below
#soggetto = "-2, 2, 1, 3, -2, -2"

func1 = ImportedPiece.notes
notes_df = corpus.batch(func=func1, kwargs={'combineUnisons': combineUnisons}, metadata=False)
func2 = ImportedPiece.melodic
melodic_df = corpus.batch(func=func2, kwargs={'kind': kind, 'end': False, 'df': notes_df}, metadata=False)
func3 = ImportedPiece.ngrams
ngrams_df = corpus.batch(func=func3, kwargs={'n': n, 'df': melodic_df}, metadata=False)
func4 = ImportedPiece.detailIndex
list_of_detail_index = corpus.batch(func=func4, kwargs={'offset': False,'df': ngrams_df}, metadata=True)

mel_corpus = pd.concat(list_of_detail_index)
comp = mel_corpus.pop("Composer")
mel_corpus['Composer'] = comp
title = mel_corpus.pop("Title")
mel_corpus["Title"] = title
mel_corpus = mel_corpus.fillna('-')
mel_corpus.columns

def _convertTuple(tup):
    out = ""
    if isinstance(tup, tuple):
        out = ', '.join(tup)
    return out

file_name_widget = widgets.Text(description='File Name:', value='input file name to save')

# df = mel_corpus

@interact
def mel_ngram_search(soggetto="", df = fixed(mel_corpus)): #soggetto="", interactive
    print(df.columns) # Gexi the Measure colume is a MultiIndex? #GT yes, it has measure number and beat inside the measure (from 0.0 to 4.5)
    df_no_tuple = df.map(_convertTuple)
    df_no_tuple.pop("Composer")
    df_no_tuple.pop("Title")
    df_no_tuple.insert(0, "Composer", mel_corpus["Composer"])
    df_no_tuple.insert(1, "Title", df["Title"])
    filtered_ngrams = df_no_tuple[df_no_tuple.apply(lambda x: x.astype(str).str.contains(soggetto).any(), axis=1)].copy()
    
    # Gexi add a colum of the name of matched soggetti
    def get_matched_indices(row):
        matched_indices = [index for index, value in row.items() if isinstance(value, str) and soggetto in value]
        return ', '.join(matched_indices)

    filtered_ngrams['Matched'] = filtered_ngrams.apply(get_matched_indices, axis=1)

    # pd.set_option('max_columns', None)

    styled_df = filtered_ngrams.fillna("-").reset_index().map(str).style.map(lambda x: "background: #ccebc4" if re.search(soggetto, x) else "")
    display(HTML(styled_df.to_html()))
    
    
    # Gexi save to csv
    def save_to_csv(button_click):
        file_name = file_name_widget.value.strip()
        os.makedirs(output_folder, exist_ok=True)
        output_file_path = os.path.join(output_folder, f"{file_name}.csv")
        normal_df = filtered_ngrams.fillna("-").reset_index().applymap(str)
        normal_df.to_csv(output_file_path, index=False)
        print(f"Already saved at {output_file_path}")

    # Gexi "save" button
    if combineUnisons:
        file_name_combineUnisons = "combT"
    else:
        file_name_combineUnisons = "combF"
    file_name_widget.value = f"{kind}_{n}_{file_name_combineUnisons}_{soggetto.replace(', ', '_').replace(' ', '')}"


    # Gexi "save" button
    display(file_name_widget)
    save_button = widgets.Button(description='Save as CSV')
    save_button.on_click(save_to_csv)
    display(save_button)

interactive(children=(Text(value='', description='soggetto'), Output()), _dom_classes=('widget-interact',))

In [49]:
folder_path = f'{project_name}/csv'

unique_numbers = set()
numbers_info = {}

for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        header_name = os.path.splitext(filename)[0]
        
        df = pd.read_csv(file_path)
        measure_column = df.iloc[:, 0]
        beat_column = df.iloc[:, 1]
        matched_column = df.iloc[:, 9]
        unique_numbers.update(measure_column.tolist())
        
        combined_values = beat_column.astype(str) + '_' + matched_column.astype(str)
        numbers_info[header_name] = dict(zip(measure_column, combined_values))

sorted_numbers = sorted(list(unique_numbers))
unique_numbers_df = pd.DataFrame(sorted_numbers, columns=['Measure'])

for header_name in numbers_info.keys():
    unique_numbers_df[header_name] = unique_numbers_df['Measure'].map(lambda x: numbers_info[header_name].get(x, '')) #GT use '-' or an emppty cell instead of '/'

unique_numbers_df = unique_numbers_df.reindex(sorted(unique_numbers_df.columns), axis=1)

output_file_path = f'{project_name}/{project_name}_Combined.csv'
unique_numbers_df.to_csv(output_file_path, index=False)
print(f"Saved at {output_file_path}")

# GT Save to Excel
output_file_path_excel = f'{project_name}/{project_name}_Combined.xlsx'
with pd.ExcelWriter(output_file_path_excel, engine='xlsxwriter') as writer:
    unique_numbers_df.to_excel(writer, index=False, sheet_name='Sheet1')
print(f"Saved Excel at {output_file_path_excel}")

Saved at mytest/mytest_Combined.csv
Saved Excel at mytest/mytest_Combined.xlsx


In [73]:
import io
import contextlib
import sys
import csv

# verovio_canvas_dict
# Define the range dictionary
height_range_dict = {
    range(0, 5): '1200',
    range(5, 10): '2000',
    range(10, 20): '2800'
}

# Function to get the value for a given number
def get_value_for_number(number):
    for key in range_dict:
        if number in key:
            return range_dict[key]
    return "Number not in any range"

def verovioPrintExampleSave(start, stop): # code via github, changed for the save function
    tk = verovio.toolkit()
    tk.loadData(fetched_mei_string)
    piece = importScore(fetched_mei_string)
    number_voices = len(piece.notes().columns)
    height = get_value_for_number(number_voices)
    tk.setScale(30)
    tk.setOptions({"pageHeight":  height, # Height in pixels
                       "pageWidth":  3000    # Width in pixels
                       })

    if stop == -1:
        meas = piece.measures()
        stop = meas.iloc[-1].tolist()[0]

    mr = str(start) + "-" + str(stop)
    mdict = {'measureRange': mr}

    if stop < start:
        print("Check the measure range, the stop measure must be equal to or greater than the start measure")
    else:
        tk.select(mdict)
        tk.redoLayout()

        # Get the number of pages and save the music as SVG
        count = tk.getPageCount()
        for c in range(1, count + 1):
            music = tk.renderToSVG(c)
            display(HTML(music))
            # file_name = f"{input_file_name}_{measure}_{beat}_{matched}.svg"
            file_name = f"{measure}.svg"
            file_path = os.path.join(output_folder, file_name)
            with open(file_path, "w") as svg_file:
                svg_file.write(music)
                print(f"SVG saved: {file_name}")


input_file_path = os.path.join(project_name, f"{project_name}_Combined.csv")
print(input_file_path)


# this should be obtained dynamically, no?  Based on path
response = requests.get("https://crimproject.org/mei/CRIM_Mass_0019_2.mei")
fetched_mei_string = response.text

output_folder = os.path.join(project_name, "svg")
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

with open(input_file_path, 'r') as file:
    reader = csv.DictReader(file)
    for row in reader:
        matched = {value for key, value in row.items() if key != 'Measure'}
        #beat = row['Beat']
        measure = row['Measure']

        mea = int(float(measure))
        print(f"Measure: {measure}, Voice(s): {', '.join(matched)}") #just measure and matched
        # piece.verovioPrintExample(mea, mea + 3)
        verovioPrintExampleSave(mea, mea + 3)

mytest/mytest_Combined.csv
Measure: 1.0, Voice(s): 1.0_Cantus


[Warning] Unsupported data.PERCENT '104'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>


SVG saved: 1.0.svg
Measure: 4.0, Voice(s): 3.0_Tenor


[Warning] Unsupported data.PERCENT '104'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>


[Warning] Unsupported data.PERCENT '104'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>


...

SVG saved: 53.0.svg
Measure: 54.0, Voice(s): 1.0_Bassus


[Warning] Unsupported data.PERCENT '104'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>


SVG saved: 54.0.svg
Measure: 55.0, Voice(s): 2.0_Altus


[Warning] Unsupported data.PERCENT '104'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported data.PERCENT '100'
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>
[Warning] Unsupported '<line>' within <measure>


SVG saved: 55.0.svg


In [74]:
import pandas as pd
import os
import webbrowser

file_path = f"{project_name}/{project_name}_Combined.csv"
output_file_path = f"{project_name}/{project_name}_Combined_withSVG.html"
data = pd.read_csv(file_path)

new_column_name = 'Sheet Music'
data.insert(loc=1, column=new_column_name, value='')

html_table = '<table border="1">\n'

for index, row in data.iterrows():
    html_table += '<tr>'
    
    if index == 0:
        for column in data.columns:
            html_table += f'<th>{column}</th>'
        html_table += '</tr>\n<tr>'
        continue
        
    
    for col_index, value in enumerate(row):
        if col_index == 1:
            number = row['Measure']
            folder_path = f"{project_name}/svg"
            svg_file = os.path.join(folder_path, f'{number}.svg')
            
            if os.path.exists(svg_file):
                with open(svg_file, 'r') as file:
                    svg_content = file.read()
                    html_table += f'<td>{svg_content}</td>'
                continue
        
        html_table += f'<td>{value}</td>'
    html_table += '</tr>\n'
html_table += '</table>'

with open(output_file_path, 'w') as html_file:
    html_file.write(html_table)
print(f"Saved as {output_file_path}")

Saved as mytest/mytest_Combined_withSVG.html
